In [1]:
import torch 
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
import pandas as pd

/home/lorenzo/Documents/projects/summarisation/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
sample_text = dataset['train']['article'][0]


In [3]:
tokenizer = AutoTokenizer.from_pretrained("./bart_cnn_finetuned")
model = AutoModelForSeq2SeqLM.from_pretrained("./bart_cnn_finetuned")
model.eval()

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=1024)
with torch.no_grad():
    out = model.generate(**inputs, num_beams=4, max_length=256, min_length=10)
print("\n" + sample_text)
print(f"\nSummary: {tokenizer.decode(out[0], skip_special_tokens=True)}")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4814.53it/s]



LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Details of ho

In [4]:
base_tokenizer = AutoTokenizer.from_pretrained("./saved_bart_model")
base_model = AutoModelForSeq2SeqLM.from_pretrained("./saved_bart_model")

base_model.eval()

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=1024)
with torch.no_grad():
    out = base_model.generate(**inputs, num_beams=4, max_length=256, min_length=10)
print("\n" + sample_text)
print(f"\nSummary: {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4710.26it/s]



LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Details of ho

In [5]:
rouge_metric = evaluate.load("rouge")

def evaluate_summaries(dataset, split="test", n_samples=20, 
                        finetuned_model=None, finetuned_tokenizer=None,
                        base_model=None, base_tokenizer=None,
                        max_input_length=512, max_gen_length=128):
    """
    Compares a fine-tuned model against a base model on n_samples from `dataset[split]`.
    Returns a dict with rouge scores for each, plus a DataFrame of per-example outputs.
    """
    samples = dataset[split].select(range(n_samples))
    articles = samples["article"]
    references = samples["highlights"]

    def generate_batch(model, tokenizer):
        preds = []
        model.eval()
        for text in articles:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input_length)
            with torch.no_grad():
                out = model.generate(**inputs, num_beams=4, max_length=max_gen_length, min_length=10)
            preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        return preds

    finetuned_preds = generate_batch(finetuned_model, finetuned_tokenizer)
    base_preds = generate_batch(base_model, base_tokenizer)

    finetuned_scores = rouge_metric.compute(predictions=finetuned_preds, references=references)
    base_scores = rouge_metric.compute(predictions=base_preds, references=references)

    comparison_df = pd.DataFrame({
        "articles": articles,
        "reference": references,
        "finetuned_summary": finetuned_preds,
        "base_summary": base_preds,
    })

    return {
        "finetuned_rouge": finetuned_scores,
        "base_rouge": base_scores,
        "comparison_df": comparison_df,
    }

In [6]:
results = evaluate_summaries(
    dataset,
    split="test",
    n_samples=10,
    finetuned_model=model, finetuned_tokenizer=tokenizer,
    base_model=base_model, base_tokenizer=base_tokenizer,
)
print("Fine-tuned:", results["finetuned_rouge"])
print("Base:", results["base_rouge"])

results["comparison_df"].head()

Fine-tuned: {'rouge1': np.float64(0.3627334692531924), 'rouge2': np.float64(0.13903623858500147), 'rougeL': np.float64(0.26898722602961134), 'rougeLsum': np.float64(0.34373743418261293)}
Base: {'rouge1': np.float64(0.4376460064205731), 'rouge2': np.float64(0.22291786742993802), 'rougeL': np.float64(0.34758640644240324), 'rougeLsum': np.float64(0.37651924470172193)}


,articles,reference,finetuned_summary,base_summary
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,Palestinian Authority becomes 123rd member of ...,Palestinian Authority becomes 123rd member of ...
1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...","Stray dog was hit by a car, buried in a field ...","Theia, a one-year-old bully breed mix, was hit..."
2,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,Mohammad Javad Zarif is the Iranian foreign mi...,Mohammad Javad Zarif is the Iranian foreign mi...
3,(CNN)Five Americans who were monitored for thr...,17 Americans were exposed to the Ebola virus w...,Five Americans exposed to Ebola in West Africa...,The five were exposed to Ebola in Sierra Leone...
4,(CNN)A Duke student has admitted to hanging a ...,Student is no longer on Duke University campus...,Student admits hanging a noose from a tree nea...,Duke student admits to hanging a noose from a ...


In [7]:
scores_df = pd.DataFrame({
    "Continue fine-tuned": results["finetuned_rouge"],
    "base": results["base_rouge"],
})
scores_df

,Continue fine-tuned,base
rouge1,0.362733,0.437646
rouge2,0.139036,0.222918
rougeL,0.268987,0.347586
rougeLsum,0.343737,0.376519
